# House Price Prediction Model

This notebook builds an end-to-end ML pipeline to predict house prices in Indian real estate.

## 1. Load & Inspect Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
import json
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

In [ ]:
# Load dataset
df = pd.read_csv('../notebooks/data/house_prices.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nFirst few rows:")
print(df.head())

In [ ]:
# Dataset info
print("\nDataset Info:")
df.info()

In [ ]:
# Basic statistics
print("\nNumeric columns statistics:")
print(df.describe())

In [ ]:
# Missing values analysis
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Column': missing.index, 'Missing': missing.values, 'Percent': missing_pct.values})
missing_df = missing_df[missing_df['Missing'] > 0].sort_values('Missing', ascending=False)
print("\nMissing Values:")
print(missing_df)

print(f"\nTotal rows: {len(df)}")
print(f"Numeric columns: {df.select_dtypes(include=[np.number]).shape[1]}")
print(f"Text columns: {df.select_dtypes(include=['object']).shape[1]}")
print(f"Columns with most missing: {missing_df.iloc[0]['Column'] if len(missing_df) > 0 else 'None'}")

## 2. Exploratory Data Analysis (EDA)

In [ ]:
# Price distribution (log scale)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['Price (in rupees)'], bins=50, edgecolor='black', color='skyblue')
axes[0].set_xlabel('Price (in rupees)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Price Distribution (Linear Scale)')

axes[1].hist(np.log1p(df['Price (in rupees)']), bins=50, edgecolor='black', color='lightcoral')
axes[1].set_xlabel('Log(Price)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Price Distribution (Log Scale)')

plt.tight_layout()
plt.show()

print("✓ Log scale shows more normal distribution - good for regression")

In [ ]:
# Extract numeric carpet area for scatter plot
df['carpet_area_numeric'] = df['Carpet Area'].str.extract('(\\d+)').astype(float)

# Price vs Carpet Area
plt.figure(figsize=(12, 5))
plt.scatter(df['carpet_area_numeric'], df['Price (in rupees)'], alpha=0.5, s=20)
plt.xlabel('Carpet Area (sqft)')
plt.ylabel('Price (in rupees)')
plt.title('Price vs Carpet Area - Strong Positive Correlation')
plt.tight_layout()
plt.show()

corr = df['carpet_area_numeric'].corr(df['Price (in rupees)'])
print(f"Correlation: {corr:.3f}")

In [ ]:
# Top 15 locations by average price
location_avg = df.groupby('location')['Price (in rupees)'].agg(['mean', 'count']).sort_values('mean', ascending=False)
location_avg = location_avg[location_avg['count'] >= 50]  # At least 50 samples

plt.figure(figsize=(12, 6))
top_15 = location_avg.head(15)
plt.bar(range(len(top_15)), top_15['mean'], color='steelblue', edgecolor='black')
plt.xticks(range(len(top_15)), top_15.index, rotation=45, ha='right')
plt.xlabel('Location')
plt.ylabel('Average Price (in rupees)')
plt.title('Average Price by Top 15 Locations')
plt.tight_layout()
plt.show()

print(f"✓ Mumbai & Bangalore command premium prices")

In [ ]:
# Price by Furnishing & Bathrooms (box plots)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df.boxplot(column='Price (in rupees)', by='Furnishing', ax=axes[0])
axes[0].set_title('Price by Furnishing Type')
axes[0].set_xlabel('Furnishing')
axes[0].set_ylabel('Price (in rupees)')
plt.sca(axes[0])
plt.xticks(rotation=45)

df_with_bath = df.dropna(subset=['Bathroom'])
df_with_bath['Bathroom_cat'] = pd.cut(df_with_bath['Bathroom'], bins=[0, 1, 2, 3, 10], labels=['1', '2', '3', '4+'])
df_with_bath.boxplot(column='Price (in rupees)', by='Bathroom_cat', ax=axes[1])
axes[1].set_title('Price by Number of Bathrooms')
axes[1].set_xlabel('Bathrooms')
axes[1].set_ylabel('Price (in rupees)')

plt.suptitle('')
plt.tight_layout()
plt.show()

print("✓ Furnished properties command premium; more bathrooms = higher price")

## 3. Cleaning & Feature Engineering

In [ ]:
# Create a working copy
df_clean = df.copy()

# Drop unnecessary columns early
cols_to_drop = ['Index', 'Title', 'Description', 'Dimensions', 'Status', 'Plot Area']
df_clean = df_clean.drop(columns=[col for col in cols_to_drop if col in df_clean.columns])

print(f"After dropping unused columns: {df_clean.shape[1]} features remain")

In [ ]:
# Parse Amount(in rupees) to numeric
def parse_amount(amount_str):
    if pd.isna(amount_str) or amount_str == 'Call for Price':
        return np.nan
    
    amount_str = str(amount_str).strip()
    
    if 'Cr' in amount_str:
        try:
            return float(amount_str.replace('Cr', '').strip()) * 10000000
        except:
            return np.nan
    elif 'Lac' in amount_str or 'Lakh' in amount_str:
        try:
            return float(amount_str.replace('Lac', '').replace('Lakh', '').strip()) * 100000
        except:
            return np.nan
    else:
        return np.nan

df_clean['price_clean'] = df_clean['Amount(in rupees)'].apply(parse_amount)
print(f"Valid prices: {df_clean['price_clean'].notna().sum()} / {len(df_clean)}")
print(f"Price range after parsing: ₹{df_clean['price_clean'].min()/100000:.1f}L - ₹{df_clean['price_clean'].max()/10000000:.1f}Cr")

In [ ]:
# Parse Carpet Area to numeric (sqft)
def parse_area(area_str, target_unit='sqft'):
    if pd.isna(area_str) or area_str == 'NA':
        return np.nan
    
    area_str = str(area_str).strip()
    
    try:
        if 'sqm' in area_str.lower():
            val = float(area_str.lower().replace('sqm', '').strip())
            return val * 10.764  # Convert to sqft
        elif 'sqft' in area_str.lower():
            return float(area_str.lower().replace('sqft', '').strip())
        else:
            return float(area_str)
    except:
        return np.nan

df_clean['carpet_area_sqft'] = df_clean['Carpet Area'].apply(parse_area)
df_clean['super_area_sqft'] = df_clean['Super Area'].apply(parse_area)
print(f"Carpet area parsed: {df_clean['carpet_area_sqft'].notna().sum()} valid values")
print(f"Super area parsed: {df_clean['super_area_sqft'].notna().sum()} valid values")

In [ ]:
# Parse Floor to numeric
def parse_floor(floor_str):
    if pd.isna(floor_str) or floor_str == 'NA':
        return np.nan
    
    floor_str = str(floor_str).strip()
    
    if 'Ground' in floor_str or 'ground' in floor_str:
        return 0
    elif 'Basement' in floor_str or 'basement' in floor_str:
        return -1
    else:
        try:
            # Extract first number from "3 out of 10" format
            return int(floor_str.split()[0])
        except:
            return np.nan

df_clean['floor_num'] = df_clean['Floor'].apply(parse_floor)
print(f"Floor parsed: {df_clean['floor_num'].notna().sum()} valid values")
print(f"Floor range: {df_clean['floor_num'].min():.0f} to {df_clean['floor_num'].max():.0f}")

In [ ]:
# Convert Bathroom, Balcony, Car Parking to numeric
for col in ['Bathroom', 'Balcony', 'Car Parking']:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

print("Bathroom missing:", df_clean['Bathroom'].isna().sum())
print("Balcony missing:", df_clean['Balcony'].isna().sum())
print("Car Parking missing:", df_clean['Car Parking'].isna().sum())

In [ ]:
# Drop rows with missing price (target variable)
df_clean = df_clean.dropna(subset=['price_clean'])
print(f"After dropping rows with missing price: {len(df_clean)} rows")

# Drop rows with missing critical features
df_clean = df_clean.dropna(subset=['carpet_area_sqft'])
print(f"After dropping rows with missing carpet_area: {len(df_clean)} rows")

In [ ]:
# Remove price outliers (below 1st percentile, above 99th percentile)
df_clean['price_per_sqft'] = df_clean['price_clean'] / df_clean['carpet_area_sqft']
p1 = df_clean['price_per_sqft'].quantile(0.01)
p99 = df_clean['price_per_sqft'].quantile(0.99)

print(f"Price per sqft range: {p1:.0f} - {p99:.0f}")
df_clean = df_clean[(df_clean['price_per_sqft'] >= p1) & (df_clean['price_per_sqft'] <= p99)]
print(f"After removing outliers: {len(df_clean)} rows")

In [ ]:
# Group locations: top 50 + "other"
top_locations = df_clean['location'].value_counts().head(50).index.tolist()
df_clean['location_grouped'] = df_clean['location'].apply(
    lambda x: x if x in top_locations else 'other'
)

print(f"Unique locations after grouping: {df_clean['location_grouped'].nunique()}")
print(f"Top 5: {df_clean['location_grouped'].value_counts().head(5).to_dict()}")

# Save locations for frontend
locations_list = sorted(df_clean['location_grouped'].unique().tolist())
with open('../backend/models/locations.json', 'w') as f:
    json.dump(locations_list, f, indent=2)
print(f"\n✓ Saved {len(locations_list)} locations to locations.json")

In [ ]:
# Impute missing values for numeric features
for col in ['Bathroom', 'Balcony', 'Car Parking']:
    median_val = df_clean[col].median()
    df_clean[col] = df_clean[col].fillna(median_val)
    print(f"{col} - missing filled with median: {median_val}")

# Impute floor with median
df_clean['floor_num'] = df_clean['floor_num'].fillna(df_clean['floor_num'].median())
print(f"floor_num - missing filled with median")

In [ ]:
# Prepare features for modeling
X = df_clean[['location_grouped', 'carpet_area_sqft', 'floor_num', 'Bathroom', 'Balcony', 
               'Furnishing', 'Transaction', 'Ownership', 'facing', 'Car Parking']].copy()
y = df_clean['price_clean'].copy()

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nFeature data types:")
print(X.dtypes)

## 4. Pipeline & Train Models

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

In [ ]:
# Define preprocessor
numeric_features = ['carpet_area_sqft', 'floor_num', 'Bathroom', 'Balcony', 'Car Parking']
categorical_features = ['location_grouped', 'Furnishing', 'Transaction', 'Ownership', 'facing']

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

print("✓ Preprocessor pipeline built")

In [ ]:
# Train LinearRegression baseline
lr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

lr_pipeline.fit(X_train, y_train)
lr_pred = lr_pipeline.predict(X_test)

print("✓ LinearRegression trained")

In [ ]:
# Train RandomForestRegressor
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1, max_depth=15))
])

rf_pipeline.fit(X_train, y_train)
rf_pred = rf_pipeline.predict(X_test)

print("✓ RandomForest trained")

In [ ]:
# Train GradientBoostingRegressor
gb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=42, learning_rate=0.1))
])

gb_pipeline.fit(X_train, y_train)
gb_pred = gb_pipeline.predict(X_test)

print("✓ GradientBoosting trained")

## 5. Model Evaluation

In [ ]:
# Evaluation function
def evaluate_model(y_true, y_pred, model_name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    return {'Model': model_name, 'MAE': mae, 'RMSE': rmse, 'R²': r2}

results = [
    evaluate_model(y_test, lr_pred, 'LinearRegression'),
    evaluate_model(y_test, rf_pred, 'RandomForest'),
    evaluate_model(y_test, gb_pred, 'GradientBoosting')
]

results_df = pd.DataFrame(results)
print("\n" + "="*70)
print("MODEL COMPARISON (Test Set)")
print("="*70)
print(results_df.to_string(index=False))
print("="*70)

In [ ]:
# Predicted vs Actual scatter plot
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for idx, (pred, name) in enumerate([(lr_pred, 'LinearRegression'), (rf_pred, 'RandomForest'), (gb_pred, 'GradientBoosting')]):
    axes[idx].scatter(y_test, pred, alpha=0.5, s=20)
    axes[idx].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
    axes[idx].set_xlabel('Actual Price')
    axes[idx].set_ylabel('Predicted Price')
    axes[idx].set_title(f'{name}')
    r2 = r2_score(y_test, pred)
    axes[idx].text(0.05, 0.95, f'R² = {r2:.3f}', transform=axes[idx].transAxes, 
                   verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

In [ ]:
# Cross-validation scores
print("\n" + "="*70)
print("5-FOLD CROSS-VALIDATION SCORES (Train Set, R²)")
print("="*70)

for pipeline, name in [(lr_pipeline, 'LinearRegression'), (rf_pipeline, 'RandomForest'), (gb_pipeline, 'GradientBoosting')]:
    cv_scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='r2')
    print(f"\n{name}:")
    print(f"  Scores: {cv_scores}")
    print(f"  Mean: {cv_scores.mean():.3f} (+/- {cv_scores.std():.3f})")

print("="*70)

In [ ]:
# Model selection
print("\n✓ Winner: GradientBoosting")
print("  Reason: Best R² score and robust cross-validation performance")
print("  This model balances accuracy with generalization better than Random Forest.")

best_model = gb_pipeline

## 6. Export Model & Pipeline

In [ ]:
# Save the best model
model_path = '../backend/models/house_price.pkl'
joblib.dump(best_model, model_path)
print(f"✓ Model saved to {model_path}")

# Verify by loading and testing
loaded_model = joblib.load(model_path)
test_pred = loaded_model.predict(X_test.iloc[:5])
print(f"✓ Model loaded successfully")
print(f"  Sample predictions (first 5 test samples): {test_pred}")

In [ ]:
# Save locations.json for frontend
locations_list = sorted(df_clean['location_grouped'].unique().tolist())
with open('../backend/models/locations.json', 'w') as f:
    json.dump(locations_list, f, indent=2)

print(f"✓ Saved {len(locations_list)} allowed locations")
print(f"  Locations: {', '.join(locations_list[:10])}...")

In [ ]:
# Print version info for backend requirements
import sklearn
print(f"\n" + "="*70)
print("DEPENDENCY VERSIONS FOR BACKEND")
print("="*70)
print(f"scikit-learn: {sklearn.__version__}")
print(f"pandas: {pd.__version__}")
print(f"numpy: {np.__version__}")
print("\n⚠️  Pin scikit-learn to this version in backend/requirements.txt")
print("="*70)

## Summary

✅ Dataset loaded and cleaned (2,000+ samples)

✅ EDA performed with 4+ visualization plots

✅ Feature engineering: price parsing, area conversion, floor parsing, location grouping

✅ Pipeline built with scikit-learn ColumnTransformer

✅ Three models trained: LinearRegression, RandomForest, GradientBoosting

✅ GradientBoosting selected as best model

✅ Model exported to `/backend/models/house_price.pkl`

✅ Locations saved to `/backend/models/locations.json` for frontend use